# Baseline 1: Logistic Regression
**DSC 148 Final Project**  
Logistic Regression assumes a linear decision boundary in feature space. Both the linearity assumption and the class-conditional feature independence are violated here, making it a useful lower bound for comparison with LightGBM.

In [4]:
#!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
SEED = 42
print('Libraries loaded.')

Libraries loaded.


## 1. Data Loading and Preprocessing

**Clean pipeline:** chronological split ' label-encode on train ' UID aggregations on train ' imputer/scaler on train.

In [ ]:
def reduce_mem(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    return df

train_trans = reduce_mem(pd.read_csv('data/train_transaction.csv'))
train_id    = reduce_mem(pd.read_csv('data/train_identity.csv'))
df = train_trans.merge(train_id, on='TransactionID', how='left')

# Temporal features (row-wise -- no leakage)
START_DATE = pd.Timestamp('2017-11-30')
dt = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['tx_hour']  = dt.dt.hour.astype('int8')
df['tx_dow']   = dt.dt.dayofweek.astype('int8')
df['tx_month'] = dt.dt.month.astype('int8')
df['card_age'] = (df['TransactionDT'] // 86400 - df['D1']).astype('float32')

print(f'Merged shape: {df.shape}')

In [ ]:
# Chronological split: 80% train / 10% val / 10% test
df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)
n       = len(df_sorted)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_df = df_sorted.iloc[:n_train].copy()
val_df   = df_sorted.iloc[n_train:n_train+n_val].copy()
test_df  = df_sorted.iloc[n_train+n_val:].copy()

y_train = train_df['isFraud'].values
y_val   = val_df['isFraud'].values
y_test  = test_df['isFraud'].values

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')
print(f'Fraud -- train : {y_train.mean()*100:.2f}%  val : {y_val.mean()*100:.2f}%  test : {y_test.mean()*100:.2f}%')

# Label-encode categoricals -- fit on train only
CAT_COLS = [
    'ProductCD','card4','card6','P_emaildomain','R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'DeviceType','DeviceInfo',
    'id_12','id_15','id_16','id_23','id_27','id_28',
    'id_29','id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38'
]
CAT_COLS = [c for c in CAT_COLS if c in train_df.columns]
for col in CAT_COLS:
    for part in [train_df, val_df, test_df]:
        part[col] = part[col].fillna('unknown').astype(str)
    cats = train_df[col].unique().tolist()
    for part in [train_df, val_df, test_df]:
        part[col] = pd.Categorical(part[col], categories=cats).codes
print(f'Label-encoded {len(CAT_COLS)} columns (fit on train; unseen -1).')

In [ ]:
# UID group features -- train partition only
for part in [train_df, val_df, test_df]:
    part['uid'] = (part['card1'].astype(str) + '_' +
                   part['addr1'].fillna(-1).astype(int).astype(str) + '_' +
                   part['card_age'].fillna(-1).round(0).astype(int).astype(str))

c_cols    = [f'C{i}' for i in range(1, 15)]
uid_stats = (train_df.groupby('uid')
               .agg(uid_tx_count=('TransactionID','count'),
                    uid_amt_mean=('TransactionAmt','mean'),
                    uid_amt_std =('TransactionAmt','std'),
                    **{f'uid_{c}_mean': (c,'mean') for c in c_cols})
               .reset_index())

fallback = {
    'uid_tx_count': 1,
    'uid_amt_mean': float(train_df['TransactionAmt'].mean()),
    'uid_amt_std' : float(train_df['TransactionAmt'].std()),
    **{f'uid_{c}_mean': float(train_df[c].mean()) for c in c_cols},
}

def attach_uid(df_part, stats, fb):
    out = df_part.merge(stats, on='uid', how='left')
    for col, val in fb.items():
        out[col] = out[col].fillna(val).astype('float32')
    return out.drop(columns=['uid'])

train_df = attach_uid(train_df, uid_stats, fallback)
val_df   = attach_uid(val_df,   uid_stats, fallback)
test_df  = attach_uid(test_df,  uid_stats, fallback)

# Feature arrays
DROP = ['TransactionID','isFraud','TransactionDT']
BASELINE_FEATURES = (
    ['TransactionAmt'] +
    ['card1','card2','card3','card5','card4','card6'] +
    ['addr1','addr2','dist1','P_emaildomain','R_emaildomain'] +
    [f'C{i}' for i in range(1,15)] +
    ['D1','D2','D4','D10','D15'] +
    ['M1','M2','M3','M4','M5','M6','M7','M8','M9','ProductCD'] +
    ['tx_hour','tx_dow','tx_month','card_age'] +
    ['uid_tx_count','uid_amt_mean','uid_amt_std'] +
    [f'uid_C{i}_mean' for i in range(1,15)]
)
BASELINE_FEATURES = [f for f in BASELINE_FEATURES if f in train_df.columns]

def _base(df_part):
    X = df_part[BASELINE_FEATURES].copy()
    X['TransactionAmt'] = np.log1p(X['TransactionAmt'])
    return X

imputer    = SimpleImputer(strategy='median')
X_train_b  = pd.DataFrame(imputer.fit_transform(_base(train_df)), columns=BASELINE_FEATURES)
X_val_b    = pd.DataFrame(imputer.transform(_base(val_df)),        columns=BASELINE_FEATURES)
X_test_b   = pd.DataFrame(imputer.transform(_base(test_df)),       columns=BASELINE_FEATURES)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_b)
X_val_sc   = scaler.transform(X_val_b)
X_test_sc  = scaler.transform(X_test_b)

print(f'Baseline features : {len(BASELINE_FEATURES)}')
print(f'Train : {X_train_sc.shape[0]:,}  |  Val : {X_val_sc.shape[0]:,}  |  Test : {X_test_sc.shape[0]:,}')

## 2. Logistic Regression

`class_weight='balanced'` rescales the loss to compensate for the 27:1 class ratio.  
Threshold is tuned on the **validation set** to maximise F1; the test set is never seen during selection.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED,
                        class_weight='balanced', C=0.1)
lr.fit(X_train_sc, y_train)

y_prob_lr_val = lr.predict_proba(X_val_sc)[:, 1]
y_prob_lr     = lr.predict_proba(X_test_sc)[:, 1]

thr_lr  = np.linspace(y_prob_lr_val.min()+1e-4, y_prob_lr_val.max()-1e-4, 300)
best_lr = float(thr_lr[int(np.argmax(
    [f1_score(y_val, (y_prob_lr_val>=t).astype(int), zero_division=0) for t in thr_lr]
))])
y_pred_lr = (y_prob_lr >= best_lr).astype(int)
print(f'Threshold (val): {best_lr:.4f}')

print('='*35)
print('Logistic Regression -- Test Set')
print('='*35)
print(classification_report(y_test, y_pred_lr, target_names=['Legitimate','Fraud']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_prob_lr):.4f}')

## 3. Results

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
auc = roc_auc_score(y_test, y_prob_lr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, color='#4878CF', label=f'LR (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Fig. 1: LR ROC Curve')
axes[0].legend()

thr_range = np.linspace(y_prob_lr_val.min()+1e-4, y_prob_lr_val.max()-1e-4, 300)
f1s  = [f1_score(y_val, (y_prob_lr_val>=t).astype(int), zero_division=0) for t in thr_range]
prec = [precision_score(y_val, (y_prob_lr_val>=t).astype(int), zero_division=0) for t in thr_range]
rec  = [recall_score(y_val, (y_prob_lr_val>=t).astype(int), zero_division=0) for t in thr_range]
axes[1].plot(thr_range, f1s,  label='F1')
axes[1].plot(thr_range, prec, label='Precision')
axes[1].plot(thr_range, rec,  label='Recall')
axes[1].axvline(best_lr, color='red', linestyle='--', linewidth=1, label=f'best thr={best_lr:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Fig. 2: LR Threshold vs. Metrics (val set)')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save metrics and probabilities for comparison in 04_proposed_lgbm.ipynb
os.makedirs('results', exist_ok=True)

metrics = {
    'model': 'Logistic Regression',
    'accuracy' : float(accuracy_score(y_test, y_pred_lr)),
    'precision': float(precision_score(y_test, y_pred_lr, zero_division=0)),
    'recall'   : float(recall_score(y_test, y_pred_lr, zero_division=0)),
    'f1'       : float(f1_score(y_test, y_pred_lr, zero_division=0)),
    'auc'      : float(roc_auc_score(y_test, y_prob_lr)),
    'threshold': best_lr,
}
with open('results/lr_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
np.save('results/lr_probs.npy', y_prob_lr)
np.save('results/y_test.npy',   y_test)
print('Saved: results/lr_metrics.json, results/lr_probs.npy, results/y_test.npy')
print(json.dumps(metrics, indent=2))